In [36]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

### 1. Загрузите данные из файла data-logistic.csv. Это двумерная выборка, целевая переменная на которой принимает значения -1 или 1.

In [37]:
df = pd.read_csv('data-logistic.csv', header=None)
y = df.iloc[:, 0].values.astype(float)
X = df.iloc[:, 1:3].values.astype(float) 

### 2. Убедитесь, что выше выписаны правильные формулы для градиентного спуска. Обратите внимание, что мы используем полноценный градиентный спуск, а не его стохастический вариант!

### 3. Реализуйте градиентный спуск для обычной и L2-регуляризованной (с коэффициентом регуляризации 10) логистической регрессии. Используйте длину шага k=0.1. В качестве начального приближения используйте вектор (0, 0).

### 4. Запустите градиентный спуск и доведите до сходимости (евклидово расстояние между векторами весов на соседних итерациях должно быть не больше 1e-5). Рекомендуется ограничить сверху число итераций десятью тысячами.

In [43]:
def train_with_step(C, step=0.1, w_init=None):
    if w_init is None:
        w = np.zeros(2)
    else:
        w = w_init.copy()
    
    reg = step * C
    max_iter = 10000
    
    for i in range(max_iter):
        scores = X @ w
        probs = 1 / (1 + np.exp(np.clip(-y * scores, -700, 700)))
        gradient = X.T @ (y * (1 - probs)) / len(y)
        
        w_new = w + step * gradient - reg * w
        
        if np.linalg.norm(w_new - w) < 1e-5:
            return w_new, True, i+1  # сошёлся
        w = w_new
    
    return w, False, max_iter

In [39]:
w0 = train(0.0)
w10 = train(10.0)

### 5. Какое значение принимает AUC-ROC на обучении без регуляризации и при ее использовании? Эти величины будут ответом на задание. В качестве ответа приведите два числа через пробел. Обратите внимание, что на вход функции roc_auc_score нужно подавать оценки вероятностей, подсчитанные обученным алгоритмом.Для этого воспользуйтесь сигмоидной функцией: a(x) = 1/(1 + exp(−w1 x1 − w2 x2 )).

In [40]:
proba = lambda w: 1 / (1 + np.exp(np.clip(-X @ w, -700, 700)))

In [41]:
auc1 = roc_auc_score(y, proba(w0))
auc2 = roc_auc_score(y, proba(w10))
ans = f"{auc1:.3f} {auc2:.3f}"
print(ans)

with open('ans.txt', 'w') as f:
    f.write(ans)

0.927 0.936


### 6. Попробуйте поменять длину шага. Будет ли сходиться алгоритм, если делать более длинные шаги? Как меняется число итераций при уменьшении длины шага?

In [47]:
print(f"{'Шаг (k)':<10} {'Сошёлся?':<12} {'Итераций':<10}")
for k in [2.0, 1.0, 0.5, 0.1, 0.05, 0.01]:
    w, conv, it = train_with_step(0.0, step=k)
    print(f"{k:.3f}\t{conv}\t\t{it}")

Шаг (k)    Сошёлся?     Итераций  
2.000	False		10000
1.000	True		32
0.500	True		60
0.100	True		244
0.050	True		431
0.010	True		1479


### 7. Попробуйте менять начальное приближение. Влияет ли оно на что-нибудь?

In [54]:
print(f"{'Старт':>12} | {'AUC':>6} | {'Итераций':>9}")

for w_start in [np.zeros(2), np.array([10.0, -5.0]), np.array([-3.0, 7.0])]:
    w, conv, it = train_with_step(0.0, step=0.1, w_init=w_start)
    auc = roc_auc_score(y, 1 / (1 + np.exp(-X @ w)))
    start_str = f"({w_start[0]:.0f}, {w_start[1]:.0f})"
    status = "ДА" if conv else "НЕТ"
    print(f"{start_str:>12} | {auc:.4f} | {it:>9}")

       Старт |    AUC |  Итераций
      (0, 0) | 0.9269 |       244
    (10, -5) | 0.9267 |       656
     (-3, 7) | 0.9269 |       502
